# Chapter 2: J-holomorphic Curves

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 2, printed pp. 17-38; PDF pp. 32-53. Sections: 2.1-2.6.

    ## Chapter Goal

    Almost complex structures, nonlinear Cauchy-Riemann equations, unique continuation, critical points, somewhere injective curves, and adjunction.

    The guiding question is:

    > What can the Cauchy-Riemann equation force before any global moduli theory is used?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| almost complex structure | field of matrices squaring to -I | J^2 + I residual |
| nonlinear Cauchy-Riemann equation | anti-holomorphic derivative | L2 residual |
| unique continuation | zero set rigidity | residual cannot vanish on an open patch only |
| somewhere injective | sample map with nonrepeated image points | rank and multiplicity |
| adjunction inequality | genus, Chern number, intersections | nonnegative defect |


## Standalone Reading Guide

    The chapter narrows attention from the global symplectic story to the analytic equation itself. In local complex coordinates the ordinary holomorphic functions are exactly the maps with zero anti-holomorphic derivative. For a general almost complex target, the equation is nonlinear because J varies with the image point, but the same diagnostic remains: the part of the derivative that disagrees with the complex structures must vanish.

Several geometric consequences are already visible locally. Unique continuation says that a solution cannot imitate another solution on a region and then split away without paying an analytic cost. Critical points are isolated rather than smeared across domains. Multiple covers must be separated from somewhere injective curves because transversality and dimension counting behave differently for the two classes. In dimension four, the adjunction philosophy records how genus, self-intersection, Chern number, and singularity data constrain one another.

The heatmap experiment uses only the standard complex structure, but it gives a reliable mental picture. The holomorphic polynomial has near-zero anti-holomorphic residual up to numerical differentiation error, while a conjugate perturbation lights up immediately. Later chapters replace this finite grid by Banach spaces, Fredholm operators, and regularity theorems.

    ## Topics In This Notebook

    - standard and perturbed almost complex structures
- Cauchy-Riemann residuals as a local diagnostic
- unique continuation and why zeros cannot be freely prescribed
- critical points, multiple covers, and somewhere injective maps
- adjunction as a bridge from local singularity data to global topology

    ## Visualization Storyboard

    - A residual heatmap compares a holomorphic polynomial with a map containing an anti-holomorphic term.
- A proof-dependency graph links local elliptic facts to global curve structure.
- A ledger tracks residual size, critical-point count, and an adjunction-style defect for a toy curve class.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-02'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['almost complex structure', 'nonlinear Cauchy-Riemann equation', 'unique continuation', 'somewhere injective', 'adjunction inequality']
CONCEPT_EDGES = [('almost complex structure', 'nonlinear Cauchy-Riemann equation'), ('nonlinear Cauchy-Riemann equation', 'unique continuation'), ('unique continuation', 'somewhere injective'), ('somewhere injective', 'adjunction inequality')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=30, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: J-holomorphic Curves')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '17-38',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in J-holomorphic Curves. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
n = 140
x = np.linspace(-1, 1, n)
y = np.linspace(-1, 1, n)
X, Y = np.meshgrid(x, y)
Z = X + 1j * Y
holomorphic = Z**3 - 0.4 * Z
perturbed = holomorphic + 0.18 * np.conj(Z) ** 2

def dbar_residual(U):
    uy, ux = np.gradient(U, y, x)
    return 0.5 * (ux + 1j * uy)

res_h = np.abs(dbar_residual(holomorphic))
res_p = np.abs(dbar_residual(perturbed))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), constrained_layout=True)
for ax, data, title in zip(axes, [res_h, res_p], ["holomorphic polynomial", "with conjugate perturbation"]):
    im = ax.imshow(data, extent=[-1, 1, -1, 1], origin="lower", cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.colorbar(im, ax=ax, shrink=0.82)
fig_path = save_matplotlib(fig, UNIT, "figures", "cauchy-riemann-residual-heatmap.png")
plt.close(fig)

check = {
    "holomorphic_mean_residual": float(res_h[5:-5, 5:-5].mean()),
    "perturbed_mean_residual": float(res_p[5:-5, 5:-5].mean()),
    "residual_ratio": float(res_p[5:-5, 5:-5].mean() / max(res_h[5:-5, 5:-5].mean(), 1e-12)),
    "passed": bool(res_p[5:-5, 5:-5].mean() > 20 * res_h[5:-5, 5:-5].mean()),
}
check_path = save_json(check, UNIT, "checks", "cauchy-riemann-residual-checks.json")
display_artifact(fig_path, width=820)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'almost complex structure', 'computational_object': 'field of matrices squaring to -I', 'check': 'J^2 + I residual'}, {'item': 'nonlinear Cauchy-Riemann equation', 'computational_object': 'anti-holomorphic derivative', 'check': 'L2 residual'}, {'item': 'unique continuation', 'computational_object': 'zero set rigidity', 'check': 'residual cannot vanish on an open patch only'}, {'item': 'somewhere injective', 'computational_object': 'sample map with nonrepeated image points', 'check': 'rank and multiplicity'}, {'item': 'adjunction inequality', 'computational_object': 'genus, Chern number, intersections', 'check': 'nonnegative defect'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Adjust the conjugate perturbation strength. Watch the residual scale linearly while the holomorphic part continues to satisfy the same local Cauchy-Riemann check.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - The anti-holomorphic derivative is the local error signal for being J-holomorphic.
- Unique continuation and isolated critical behavior are analytic rigidity statements.
- Somewhere injective curves are the basic objects for transversality arguments.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'cauchy-riemann-residual-heatmap.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'cauchy-riemann-residual-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'cauchy-riemann-residual-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
